# 🎨 Two-Stage Color Classification using K-Nearest Neighbors (KNN)
### A Simple, Intuitive Machine Learning Framework for Color Recognition
**Designed for Graduate Research & Academic Presentation**

**Dataset:** [KopolArt Color Collection](https://kopolart.ir/colors), curated by
**Ali Yousefi (KopolArt)** and licensed under **CC BY 4.0**. Required attribution:
**کپل‌آرت – Ali Yousefi** with a direct link to <https://kopolart.ir/colors>.
See `data/README.md` for the full schema and reuse terms.

---

### 📌 Research Motivation & Problem Statement
In computer vision and human-computer interaction, color naming faces a major challenge:
1. **The Specific Naming Bottleneck (High Class Sparsity):**
   - Real-world color datasets contain hundreds of specific shade names (e.g., `Dark Navy Blue`, `Light Sky Blue`, `Mint Green`, `Olive Espresso`).
   - In our dataset of **608 samples**, there are **313 unique color names**!
   - Many shades only have **1 or 2 examples**. 
   - A traditional 1-stage classifier attempting to classify all 313 classes directly achieves only **~12% (1-NN) to ~25% (tuned K=15)** exact-match accuracy because slight variations (e.g., predicting `Navy Blue` instead of `Dark Navy`) are penalized as 100% wrong.

2. **The Proposed Solution (Two-Stage Hierarchical Classification):**
   - **Stage 1 (Coarse Classification):** Use **K-Nearest Neighbors (KNN)** to classify the broad **Base Color Family** (e.g., Red, Blue, Green, Yellow, Orange, Purple, Pink, Brown, Gray, White, Black).
   - **Stage 2 (Fine Retrieval):** Within the predicted color family, find the closest specific color using **Nearest Neighbor search (1-NN)** in color space to retrieve the exact English name and Persian translation.

```
       Input RGB / Hex Code
                 │
                 ▼
       ┌───────────────────┐
       │   Pre-processing  │ (StandardScaler on R, G, B)
       └─────────┬─────────┘
                 │
                 ▼
     ┌───────────────────────┐
     │ Stage 1: KNN Model    │ ➔ Predicts Base Color Family (e.g., "Blue")
     └───────────┬───────────┘
                 │
                 ▼
     ┌───────────────────────┐
     │ Stage 2: Nearest      │ ➔ Finds closest exact shade in that family
     │ Neighbor Retrieval    │   (e.g., "Dark Navy Blue" / "سرمه‌ای سیر")
     └───────────────────────┘
```

**Key Advantages:**
- Simple, standard scikit-learn syntax.
- Raises **base-family** accuracy to **61.48%** (Top-3: **79.51%**), with a pipeline that is easy to explain.
- Exact-shade naming remains hard: **14.75%** end-to-end vs **11.48%** for a fair single-stage 1-NN baseline (a tuned K=15 fine KNN reaches **24.59%**).
- Easy to explain in a PhD presentation or seminar!


In [1]:
# 1. Import Essential Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

# Visual style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

print("✅ All required libraries imported successfully!")


✅ All required libraries imported successfully!


---
## 2. Load and Inspect the Dataset
We load `colors.csv` which contains 608 color samples with their RGB values, base color categories, and Persian translations.

In [2]:
# Load dataset
dataset_path = 'data/colors.csv' if os.path.exists('data/colors.csv') else '../data/colors.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(f"Total color samples:                 {len(df)}")
print(f"Number of raw specific names:        {df['EnglishName'].nunique()} classes (Too sparse for 1-stage ML!)")
print(f"Number of normalized base families:  {df['Base_English'].nunique()} classes (Well-balanced for KNN)")

# Display first 5 rows with essential columns
display_cols = ['EnglishName', 'Persian Name', 'HexCode (24 bit)', 'Red (8 bit)', 'Green (8 bit)', 'Blue (8 bit)', 'Base_English']
df[display_cols].head()


Dataset Shape: 608 rows, 18 columns

Total color samples:                 608
Number of raw specific names:        313 classes (Too sparse for 1-stage ML!)
Number of normalized base families:  33 classes (Well-balanced for KNN)


,EnglishName,Persian Name,HexCode (24 bit),Red (8 bit),Green (8 bit),Blue (8 bit),Base_English
0,Dark Green,سبز تیره,#388E3C,56,142,60,Green
1,Green,سبز,#8BC34A,139,195,74,Green
2,Yellow,زرد,#FEED30,254,237,48,Yellow
3,Orange,نارنجی,#FF6A38,255,106,56,Orange
4,Dark Violet,بنفش تیره,#B000B9,176,0,185,Purple


---
## 3. Exploratory Data Analysis (EDA)
Let's visualize the distribution of base color families and inspect the 3D RGB color space.

In [3]:
# 1. Bar chart of Top 12 Base Color Categories
top_bases = df['Base_English'].value_counts().head(12)

plt.figure(figsize=(11, 4))
sns.barplot(x=top_bases.index, y=top_bases.values, palette='viridis')
plt.title('Distribution of Top Base Color Families', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Base Color Category', fontsize=11)
plt.ylabel('Number of Samples', fontsize=11)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 2. 3D Color Space Scatter Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Normalize RGB values to [0, 1] for actual plot colors
colors_rgb = df[['Red (8 bit)', 'Green (8 bit)', 'Blue (8 bit)']].values / 255.0

ax.scatter(
    df['Red (8 bit)'],
    df['Green (8 bit)'],
    df['Blue (8 bit)'],
    c=colors_rgb,
    s=35,
    edgecolors='none',
    alpha=0.85
)

ax.set_title('3D RGB Color Cube (Color Distribution)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Red (0-255)')
ax.set_ylabel('Green (0-255)')
ax.set_zlabel('Blue (0-255)')
plt.tight_layout()
plt.show()


---
## 4. Feature Extraction & Train-Test Split
We use the 3 color channels `[Red, Green, Blue]` as features ($X$) and the `Base_English` category as our target ($y$).
- **Train/Test Split:** 80% for training (486 samples), 20% for testing (122 samples).
- **StandardScaler:** Centers and scales RGB values so Euclidean distances are well-balanced.

In [4]:
# Define features (X) and target (y)
feature_cols = ['Red (8 bit)', 'Green (8 bit)', 'Blue (8 bit)']
X = df[feature_cols].values
y = df['Base_English'].values

# Split into 80% Train and 20% Test sets
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index.values, test_size=0.20, random_state=42
)

# Exact shade labels are used only for the end-to-end evaluation below.
y_fine = df['EnglishName'].values
y_fine_train, y_fine_test = y_fine[idx_train], y_fine[idx_test]

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep a reference to the training dataframe for Stage 2 retrieval
train_df = df.iloc[idx_train].reset_index(drop=True)

print(f"Training samples: {len(X_train)}  (80%)")
print(f"Testing samples:  {len(X_test)}  (20%)")
print("Features scaled successfully!")


Training samples: 486  (80%)
Testing samples:  122  (20%)
Features scaled successfully!


---
## 5. Hyperparameter Tuning: Finding the Optimal $K$

The choice of $K$ in K-Nearest Neighbors represents the bias-variance tradeoff:
- **Too small ($K=1$):** High variance, sensitive to noisy or overlapping color borders.
- **Too large ($K > 15$):** High bias, oversmooths small color classes.

To avoid test-set leakage, $K$ is selected with **3-fold cross-validation on the
training set only**. The test set is used once, after $K$ has been fixed, for the
final evaluation.

In [5]:
# Select K with 3-fold cross-validation on the training set only.
k_values = [1, 2, 3, 4, 5, 7, 9, 11, 15, 21]
cv_accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k, weights='distance')
    score = cross_val_score(model, X_train_scaled, y_train, cv=3).mean()
    cv_accuracies.append(score)

best_k_idx = int(np.argmax(cv_accuracies))
best_k = k_values[best_k_idx]
best_cv_acc = cv_accuracies[best_k_idx]

# Plot K vs cross-validation accuracy
plt.figure(figsize=(9, 4.5))
plt.plot(k_values, [a * 100 for a in cv_accuracies], marker='o',
         color='#2b5c8f', linewidth=2, markersize=7)
plt.scatter([best_k], [best_cv_acc * 100], color='red', s=120, zorder=5,
            label=f'Selected K={best_k} ({best_cv_acc*100:.1f}% CV)')
plt.title('Hyperparameter Tuning: K Value vs 3-Fold CV Accuracy',
          fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Number of Neighbors (K)', fontsize=11)
plt.ylabel('Mean CV Accuracy (%)', fontsize=11)
plt.xticks(k_values)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Optimal K selected by 3-fold CV: {best_k} "
      f"(CV accuracy: {best_cv_acc * 100:.2f}%)")

Optimal K selected by 3-fold CV: 15 (CV accuracy: 56.17%)


---
## 6. Stage 1: Train KNN Model & Comprehensive Performance Evaluation
We train our final Stage 1 classifier using the CV-selected $K$ with distance weighting (`weights='distance'`).
To provide an academic, graduate-level assessment, we evaluate:
- **Confusion Matrix**
- **Overall Accuracy** & **Top-3 Accuracy**
- **Precision** (Positive Predictive Value)
- **Recall / Sensitivity** (True Positive Rate - TPR)
- **Specificity** (True Negative Rate - TNR)
- **F1-Score** (Harmonic Mean of Precision & Recall)
- **Macro and Weighted Averages**

In [6]:
# 1. Train final Stage 1 KNN model
knn_base = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
knn_base.fit(X_train_scaled, y_train)

# 2. Predict on test set
pred_base = knn_base.predict(X_test_scaled)
stage1_acc = accuracy_score(y_test, pred_base)

# 3. Calculate Top-3 accuracy
distances, indices = knn_base.kneighbors(X_test_scaled, n_neighbors=3)
top3_correct = sum(y_test[i] in y_train[indices[i]] for i in range(len(y_test)))
top3_acc = top3_correct / len(y_test)

# 4. Multi-Class Evaluation Metrics (One-vs-Rest decomposition)
test_classes = sorted(list(set(y_test) | set(pred_base)))
cm_full = confusion_matrix(y_test, pred_base, labels=test_classes)

class_metrics = []
for i, cls in enumerate(test_classes):
    tp = cm_full[i, i]
    fn = cm_full[i, :].sum() - tp
    fp = cm_full[:, i].sum() - tp
    tn = cm_full.sum() - (tp + fn + fp)
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sens = rec  # Sensitivity is mathematically identical to Recall (True Positive Rate)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    class_acc = (tp + tn) / cm_full.sum()
    support = tp + fn
    
    class_metrics.append({
        'Class': cls,
        'Support': support,
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Accuracy': class_acc,
        'Precision': prec,
        'Sensitivity (Recall)': sens,
        'Specificity': spec,
        'F1-Score': f1
    })

df_metrics = pd.DataFrame(class_metrics)
df_active_metrics = df_metrics[df_metrics['Support'] > 0].copy()

# Compute Macro and Weighted Averages across all test classes
macro_prec = df_active_metrics['Precision'].mean()
macro_sens = df_active_metrics['Sensitivity (Recall)'].mean()
macro_spec = df_active_metrics['Specificity'].mean()
macro_f1 = df_active_metrics['F1-Score'].mean()

weights = df_active_metrics['Support'] / df_active_metrics['Support'].sum()
weighted_prec = (df_active_metrics['Precision'] * weights).sum()
weighted_sens = (df_active_metrics['Sensitivity (Recall)'] * weights).sum()
weighted_spec = (df_active_metrics['Specificity'] * weights).sum()
weighted_f1 = (df_active_metrics['F1-Score'] * weights).sum()

print("=" * 72)
print("📊 STAGE 1 COMPREHENSIVE PERFORMANCE SUMMARY")
print(f"Dataset Base Families: {df['Base_English'].nunique()} | Test Set Ground-Truth Classes: {len(df_active_metrics)} | Evaluated Classes: {len(test_classes)}")
print("=" * 72)
print(f"🎯 Overall Test Accuracy (Top-1): {stage1_acc * 100:.2f}%")
print(f"🌟 Top-3 Accuracy:                {top3_acc * 100:.2f}%")
print("-" * 72)
print(f"📈 Macro Precision:               {macro_prec * 100:.2f}%  |  Weighted: {weighted_prec * 100:.2f}%")
print(f"🔍 Macro Sensitivity (Recall):    {macro_sens * 100:.2f}%  |  Weighted: {weighted_sens * 100:.2f}% (Identical to Top-1 Accuracy)")
print(f"🛡️ Macro Specificity:              {macro_spec * 100:.2f}%  |  Weighted: {weighted_spec * 100:.2f}%")
print(f"⚖️ Macro F1-Score:                 {macro_f1 * 100:.2f}%  |  Weighted: {weighted_f1 * 100:.2f}%")
print("=" * 72)

# Display Top Base Categories in a formatted table
top_classes_display = df_active_metrics.sort_values(by='Support', ascending=False).head(10).copy()
format_cols = ['Accuracy', 'Precision', 'Sensitivity (Recall)', 'Specificity', 'F1-Score']
for col in format_cols:
    top_classes_display[col] = top_classes_display[col].apply(lambda v: f"{v * 100:.1f}%")

display_cols = ['Class', 'Support', 'TP', 'FP', 'FN', 'TN', 'Accuracy', 'Precision', 'Sensitivity (Recall)', 'Specificity', 'F1-Score']
print("\n📋 Detailed Per-Class Metrics Table (Top Base Categories):")
print(top_classes_display[display_cols].to_string(index=False))


📊 STAGE 1 COMPREHENSIVE PERFORMANCE SUMMARY
Dataset Base Families: 33 | Test Set Ground-Truth Classes: 24 | Evaluated Classes: 25
🎯 Overall Test Accuracy (Top-1): 61.48%
🌟 Top-3 Accuracy:                79.51%
------------------------------------------------------------------------
📈 Macro Precision:               44.31%  |  Weighted: 61.75%
🔍 Macro Sensitivity (Recall):    44.44%  |  Weighted: 61.48% (Identical to Top-1 Accuracy)
🛡️ Macro Specificity:              98.34%  |  Weighted: 97.80%
⚖️ Macro F1-Score:                 42.71%  |  Weighted: 59.89%

📋 Detailed Per-Class Metrics Table (Top Base Categories):
 Class  Support  TP  FP  FN  TN Accuracy Precision Sensitivity (Recall) Specificity F1-Score
Orange       15  11   1   4 106    95.9%     91.7%                73.3%       99.1%    81.5%
  Blue       12   7   4   5 106    92.6%     63.6%                58.3%       96.4%    60.9%
 White       12  12   3   0 107    97.5%     80.0%               100.0%       97.3%    88.9%
 Green  

---
## 7. Mathematical & Conceptual Foundations of Classification Metrics

In multi-class machine learning, evaluating a model requires a multi-dimensional perspective beyond simple accuracy. Here is the formal mathematical framework used in our analysis:

### 1. Confusion Matrix (ماتریس درهم‌ریختگی)
The confusion matrix $C$ maps actual ground-truth classes (rows) against predicted classes (columns). For any specific class $k$ in a One-vs-Rest decomposition:
- **True Positives ($TP_k$):** Instances of class $k$ correctly classified as $k$ ($C_{k,k}$).
- **False Positives ($FP_k$):** Instances of other classes falsely predicted as $k$ ($\sum_{j \neq k} C_{j,k}$).
- **False Negatives ($FN_k$):** Instances of class $k$ falsely predicted as other classes ($\sum_{j \neq k} C_{k,j}$).
- **True Negatives ($TN_k$):** Instances of other classes correctly identified as not belonging to $k$ ($\sum_{i \neq k} \sum_{j \neq k} C_{i,j}$).

---

### 2. Metric Formulations & Experimental Results

| Metric | Mathematical Formula | Output (Test Set $n=122$) | Conceptual Interpretation |
| :--- | :---: | :---: | :--- |
| **Accuracy (صحت)** | $$\frac{\sum_k TP_k}{N_{\text{total}}}$$ | **61.48% (Top-1)<br>79.51% (Top-3)** | Ratio of correct predictions over total instances. |
| **Precision (دقت)** | $$\frac{TP_k}{TP_k + FP_k}$$ | **44.31% (Macro)<br>61.75% (Weighted)** | **Exactness:** When the model claims a color is class $k$, how often is it correct? Measures freedom from false alarms. |
| **Recall / Sensitivity (حساسیت)** | $$\frac{TP_k}{TP_k + FN_k}$$ | **44.44% (Macro)<br>61.48% (Weighted)** | **Completeness / True Positive Rate (TPR):** Fraction of actual class $k$ samples captured. *(Sensitivity $\equiv$ Recall).* |
| **Specificity (ویژگی)** | $$\frac{TN_k}{TN_k + FP_k}$$ | **98.34% (Macro)<br>97.80% (Weighted)** | **True Negative Rate (TNR):** Rejection of non-$k$ samples. *(Note: Naturally high in multi-class One-vs-Rest).* |
| **F1-Score (امتیاز اف-یک)** | $$2 \times \frac{\text{Precision}_k \times \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$ | **42.71% (Macro)<br>59.89% (Weighted)** | **Harmonic Mean:** Balances Precision and Recall, avoiding distortion on rare classes. |

---

### 3. Critical Analytical & Statistical Insights:

1. **Clarification on Class Counts:**
   - The entire dataset contains **33 Base Color Families** (`df['Base_English'].nunique()`).
   - In our 20% test split ($n=122$ samples), **24 distinct base classes** have ground-truth samples (`len(df_active_metrics)`).
   - Across ground-truth and model predictions, **25 unique classes** are evaluated (`len(test_classes)`).

2. **Sensitivity Breakdown across Color Families:**
   - **White:** Achieves **100.0% Sensitivity** (12/12 samples correctly identified).
   - **Red:** Achieves **83.3% Sensitivity** (5/6 samples detected).
   - **Yellow:** Achieves **77.8% Sensitivity** (7/9 samples detected).
   - **Orange:** Achieves **73.3% Sensitivity** (11/15 samples detected).
   - **Green:** Achieves **70.0% Sensitivity** (7/10 samples detected).
   - Sensitivity and Recall are mathematically identical terms ($TPR = \frac{TP}{TP + FN}$).

3. **Academic Reality Check on Specificity (~98.34%):**
   - In a One-vs-Rest setup with 24–25 classes, for any single class (e.g., Blue), the "negative" pool is formed by combining all other 23+ classes (~96% of the dataset).
   - Consequently, True Negatives ($TN$) overwhelmingly dwarf False Positives ($FP$), making Specificity naturally close to 98% almost automatically.
   - **Caution:** High specificity is an expected mathematical byproduct of multi-class One-vs-Rest, **not** proof of exceptional performance. Evaluative weight must primarily rest on **Precision** and **Sensitivity (Recall)**.

4. **Mathematical Identity: Weighted Recall $\equiv$ Overall Accuracy:**
   $$\text{Weighted Recall} = \sum_{k} \frac{N_k}{N_{\text{total}}} \cdot \text{Recall}_k = \sum_{k} \frac{N_k}{N_{\text{total}}} \cdot \frac{TP_k}{N_k} = \frac{\sum_k TP_k}{N_{\text{total}}} \equiv \text{Overall Accuracy} = \mathbf{61.48\%}$$
   Because each class's recall is weighted by its sample proportion $N_k / N$, Weighted Recall and Overall Top-1 Accuracy are mathematically identical (both **61.48%**).


---
## 8. Visual Confusion Matrix & Metric Analysis
We visualize:
1. **Absolute Confusion Matrix Heatmap:** Raw sample counts showing correct hits along the diagonal and error leakage off-diagonal.
2. **Normalized Confusion Matrix Heatmap:** Row-normalized, displaying the exact **Recall / Sensitivity** percentage on the diagonal.
3. **Multi-Metric Comparison Bar Chart:** Direct comparison of **Precision**, **Sensitivity (Recall)**, **Specificity**, and **F1-Score** across the primary color families.

In [7]:
# 1. Confusion Matrix Heatmaps (Counts & Normalized Recall)
top_classes = pd.Series(y_test).value_counts().head(8).index.tolist()
mask = [label in top_classes for label in y_test]
y_test_top = np.array(y_test)[mask]
pred_top = np.array(pred_base)[mask]

cm_top = confusion_matrix(y_test_top, pred_top, labels=top_classes)
cm_norm = cm_top.astype('float') / np.maximum(cm_top.sum(axis=1)[:, np.newaxis], 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Absolute Counts
sns.heatmap(cm_top, annot=True, fmt='d', cmap='Blues',
            xticklabels=top_classes, yticklabels=top_classes, ax=axes[0], cbar=True)
axes[0].set_title('Stage 1 Confusion Matrix (Absolute Counts)', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Predicted Base Color', fontsize=11)
axes[0].set_ylabel('True Base Color', fontsize=11)

# Panel B: Normalized (Sensitivity / Recall)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=top_classes, yticklabels=top_classes, ax=axes[1], cbar=True)
axes[1].set_title('Normalized Confusion Matrix (Recall / Sensitivity)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Predicted Base Color', fontsize=11)
axes[1].set_ylabel('True Base Color', fontsize=11)

plt.tight_layout()
plt.show()

# 2. Grouped Bar Chart of Precision, Recall (Sensitivity), Specificity, and F1-Score
chart_df = df_active_metrics.sort_values(by='Support', ascending=False).head(8)
chart_classes = chart_df['Class'].tolist()

x = np.arange(len(chart_classes))
width = 0.20

plt.figure(figsize=(13, 5))
plt.bar(x - 1.5 * width, chart_df['Precision'] * 100, width, label='Precision', color='#1f77b4', alpha=0.9)
plt.bar(x - 0.5 * width, chart_df['Sensitivity (Recall)'] * 100, width, label='Sensitivity (Recall)', color='#2ca02c', alpha=0.9)
plt.bar(x + 0.5 * width, chart_df['Specificity'] * 100, width, label='Specificity', color='#9467bd', alpha=0.9)
plt.bar(x + 1.5 * width, chart_df['F1-Score'] * 100, width, label='F1-Score', color='#ff7f0e', alpha=0.9)

plt.title('Stage 1 Multi-Class Performance Comparison across Top Base Colors', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Base Color Category', fontsize=11)
plt.ylabel('Score (%)', fontsize=11)
plt.xticks(x, chart_classes, rotation=20, fontsize=10.5)
plt.ylim(0, 115)
plt.legend(loc='lower right', frameon=True, fontsize=10.5)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


---
## 9. Stage 2: Fine-Grained Shade Retrieval (Nearest Neighbor)
Once the base color family is predicted in Stage 1, Stage 2 performs a nearest-neighbor search **only within that family** in the training dataset.

This achieves two major advantages:
1. **Contextual constraint:** A query predicted as `Blue` will never be matched with a green or purple shade.
2. **Bilingual name recovery:** We retrieve both the exact English shade name and Persian translation seamlessly!

In [8]:
def hex_to_rgb(hex_code: str) -> list:
    """Converts a hex color code (e.g. '#388E3C') into an RGB list [R, G, B]."""
    hex_code = hex_code.strip().lstrip('#')
    return [int(hex_code[i:i+2], 16) for i in (0, 2, 4)]

def predict_two_stage_color(hex_code: str) -> dict:
    """
    Two-Stage Color Classification Pipeline:
      - Stage 1: Predicts broad Base Color category using KNN.
      - Stage 2: Retrieves closest exact shade and Persian translation using Nearest Neighbor.
    """
    r, g, b = hex_to_rgb(hex_code)
    
    # 1. Scale input features
    rgb_scaled = scaler.transform([[r, g, b]])
    
    # 2. Stage 1: Predict Base Color Family
    predicted_base = knn_base.predict(rgb_scaled)[0]
    
    # 3. Stage 2: Filter candidates belonging to that base color
    candidates = train_df[train_df['Base_English'] == predicted_base]
    if candidates.empty:
        candidates = train_df  # Fallback to entire dataset if category is empty
    
    # 4. Find nearest shade by Euclidean distance in RGB space
    cand_rgb = candidates[['Red (8 bit)', 'Green (8 bit)', 'Blue (8 bit)']].values
    distances = np.linalg.norm(cand_rgb - [r, g, b], axis=1)
    
    best_idx = np.argmin(distances)
    match_row = candidates.iloc[best_idx]
    
    return {
        'input_hex': hex_code if hex_code.startswith('#') else f'#{hex_code}',
        'input_rgb': (r, g, b),
        'stage1_base_color': predicted_base,
        'stage2_exact_en': match_row['EnglishName'],
        'stage2_exact_fa': match_row['Persian Name'],
        'matched_hex': match_row['HexCode (24 bit)'],
        'color_distance': round(float(distances[best_idx]), 2)
    }

# Quick test with a dark navy sample
sample_result = predict_two_stage_color('#142F43')
for k, v in sample_result.items():
    print(f"{k}: {v}")


input_hex: #142F43
input_rgb: (20, 47, 67)
stage1_base_color: Navy Blue
stage2_exact_en: Dark Navy Blue
stage2_exact_fa: سرمه‌ای سیر
matched_hex: #142F43
color_distance: 0.0


In [9]:
# ---------------------------------------------------------------------------
# Baseline vs end-to-end comparison on the test set
# ---------------------------------------------------------------------------
# Fair single-stage baseline: 1-NN over ALL training shades in raw RGB space.
# This isolates the effect of the two-stage family restriction, because Stage 2
# also performs 1-NN retrieval in raw RGB.
knn_fine_1nn = KNeighborsClassifier(n_neighbors=1)
knn_fine_1nn.fit(X_train, y_fine_train)
pred_fine_1nn = knn_fine_1nn.predict(X_test)

_, fine1_neighbors = knn_fine_1nn.kneighbors(X_test, n_neighbors=3)
fine1_exact = accuracy_score(y_fine_test, pred_fine_1nn)
fine1_top3 = np.mean([
    y_fine_test[i] in y_fine_train[fine1_neighbors[i]]
    for i in range(len(y_fine_test))
])

# Tuned single-stage fine KNN baseline (313 shade classes)
knn_fine = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
knn_fine.fit(X_train_scaled, y_fine_train)
pred_fine = knn_fine.predict(X_test_scaled)
fine_exact = accuracy_score(y_fine_test, pred_fine)

_, fine_neighbors = knn_fine.kneighbors(X_test_scaled, n_neighbors=3)
fine_top3 = np.mean([
    y_fine_test[i] in y_fine_train[fine_neighbors[i]]
    for i in range(len(y_fine_test))
])

# End-to-end two-stage exact-shade retrieval on the test set
test_true_en = df.iloc[idx_test]['EnglishName'].values
two_stage_exact = 0
two_stage_top3 = 0

for i, row_idx in enumerate(idx_test):
    candidates = train_df[train_df['Base_English'] == pred_base[i]]
    if candidates.empty:
        candidates = train_df

    distances = np.linalg.norm(candidates[feature_cols].values - X[row_idx], axis=1)
    order = np.argsort(distances)[:3]
    names = candidates.iloc[order]['EnglishName'].tolist()

    two_stage_exact += names[0] == test_true_en[i]
    two_stage_top3 += test_true_en[i] in names

two_stage_exact /= len(idx_test)
two_stage_top3 /= len(idx_test)

print("Stage 1 base KNN (33 color families)")
print(f"  Base Top-1 : {stage1_acc * 100:.2f}%  |  Base Top-3 : {top3_acc * 100:.2f}%")
print()
print("Single-stage fine 1-NN baseline (313 shade classes, raw RGB)")
print(f"  Exact Top-1: {fine1_exact * 100:.2f}%  |  Exact Top-3: {fine1_top3 * 100:.2f}%")
print()
print(f"Single-stage fine KNN baseline (313 shade classes, K={best_k}, distance-weighted)")
print(f"  Exact Top-1: {fine_exact * 100:.2f}%  |  Exact Top-3: {fine_top3 * 100:.2f}%")
print()
print("End-to-end two-stage pipeline (Stage 1 + RGB retrieval)")
print(f"  Exact Top-1: {two_stage_exact * 100:.2f}%  |  Exact Top-3: {two_stage_top3 * 100:.2f}%")


Stage 1 base KNN (33 color families)
  Base Top-1 : 61.48%  |  Base Top-3 : 79.51%

Single-stage fine 1-NN baseline (313 shade classes, raw RGB)
  Exact Top-1: 11.48%  |  Exact Top-3: 31.15%

Single-stage fine KNN baseline (313 shade classes, K=15, distance-weighted)
  Exact Top-1: 24.59%  |  Exact Top-3: 31.15%

End-to-end two-stage pipeline (Stage 1 + RGB retrieval)
  Exact Top-1: 14.75%  |  Exact Top-3: 29.51%


---
## 10. Interactive Testing & Visual Demo Cards
Let's test our two-stage model on diverse colors across the warm, cold, and neutral spectrums and display visual swatches.

In [10]:
# Diverse test colors across Warm, Cold, and Neutral spectrums
test_palette = [
    '#FF1818',  # Bright Red
    '#F4722B',  # Dark Orange
    '#EAC100',  # Golden / Amber Yellow
    '#388E3C',  # Dark Green
    '#10375C',  # Dark Navy Blue
    '#7C3E66',  # Plum Purple
    '#ECECEC',  # Light Gray / Off-white
    '#1B1B1B',  # Charcoal Black
]

results = [predict_two_stage_color(c) for c in test_palette]

# Visual Presentation Cards
fig, axes = plt.subplots(len(results), 1, figsize=(10, len(results) * 0.9))

for i, (ax, res) in enumerate(zip(axes, results)):
    ax.set_facecolor('#ffffff')
    
    # Draw input color box
    rect_in = plt.Rectangle((0.02, 0.15), 0.10, 0.7, color=res['input_hex'], ec='#333333', lw=1.5)
    ax.add_patch(rect_in)
    
    # Draw matched color box
    rect_out = plt.Rectangle((0.15, 0.15), 0.10, 0.7, color=res['matched_hex'], ec='#333333', lw=1.5)
    ax.add_patch(rect_out)
    
    # Text annotations
    text_info = (
        f"Input: {res['input_hex']}   ➔   "
        f"Stage 1 (Base): {res['stage1_base_color']}   |   "
        f"Stage 2: {res['stage2_exact_en']} ({res['stage2_exact_fa']})   "
        f"[Dist: {res['color_distance']}]"
    )
    ax.text(0.28, 0.45, text_info, fontsize=10.5, verticalalignment='center', fontweight='normal')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle('🎨 Two-Stage Model Predictions on Diverse Color Samples', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()


---
## 11. Academic Conclusions & Presentation Summary

### 📊 Performance Comparison

| Approach | Target | Top-1 | Top-3 |
| :--- | :---: | :---: | :---: |
| **Stage 1 base KNN** | 33 base color families | **61.48%** | **79.51%** |
| Single-stage fine 1-NN (raw RGB) | 313 shade names | **11.48%** | **31.15%** |
| Single-stage fine KNN (K=15, distance) | 313 shade names | **24.59%** | **31.15%** |
| **End-to-end two-stage pipeline** | 33 families + exact shade | **14.75%** | **29.51%** |

**How to read this:** the 61.48% / 79.51% figures are **Stage 1 base-family accuracy**
(33 classes), not exact-shade accuracy. Against a fair 1-NN baseline in the same raw
RGB retrieval space, the two-stage restriction improves exact Top-1 from 11.48% to
14.75% (+3.27 points) but loses 1.64 points on Top-3, because an incorrect Stage 1
family prevents Stage 2 from finding the right shade. A tuned multi-neighbor fine KNN
still achieves the best exact Top-1 (24.59%), so the two-stage design should be
presented as an interpretable coarse-to-fine pipeline, not as a 61% exact-shade
classifier.

### 🎓 Key Takeaways for Your Presentation

1. **Hierarchical Problem Decomposition:**
   Breaking a 300+ class problem into coarse family grouping (Stage 1) plus fine
   retrieval (Stage 2) is a standard coarse-to-fine pattern in machine learning.
2. **Leakage-Free Model Selection:**
   K was selected by 3-fold cross-validation on the training set, and the test set
   was used only once for the final report.
3. **Geometric Clustered Nature of Color:**
   Human perception of color maps closely to distance in 3D RGB space, which makes
   distance-weighted KNN a natural and interpretable choice.
4. **Honest Statistical Interpretation:**
   Report Stage 1 base accuracy separately from exact-shade accuracy. High
   Specificity (~98%) is mathematically expected in multi-class One-vs-Rest
   evaluation; it is not by itself evidence of a superior model.

### Dataset
This notebook uses the **KopolArt Color Collection** (<https://kopolart.ir/colors>),
curated by **Ali Yousefi (KopolArt)** and licensed under **CC BY 4.0**. If you reuse
the dataset, credit **کپل‌آرت – Ali Yousefi** and link to the source.